# Advanced JSON Decoding in Python — Problems with Solutions

This notebook focuses on **advanced, practical JSON decoding** using Python's standard-library `json` module.

You will work with:

- `json.JSONDecoder`
- `object_hook`
- `object_pairs_hook`
- `parse_float`, `parse_int`, and `parse_constant`
- custom tagged objects
- `Decimal`
- recursive/nested decoding
- duplicate-key detection
- schema-style validation
- `raw_decode`
- streaming / concatenated JSON values
- defensive decoding
- round-trip serialization/deserialization
- dataclasses, `datetime`, `UUID`, sets, tuples, and domain objects

> Best-practice note: for most custom object reconstruction, prefer `object_hook`
> over overriding `JSONDecoder.decode`. Override `decode` only when you truly need
> control over the complete JSON string or whole-document behavior.


In [1]:
import json
import math
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from decimal import Decimal
from pprint import pprint
from uuid import UUID, uuid4


## Warm-up: What `JSONDecoder` actually gives you

A custom `JSONDecoder` can customize parsing in several ways.

The most useful extension points are usually:

- `object_hook`: called for every decoded JSON object (`dict`), from inner objects outward.
- `object_pairs_hook`: receives ordered `(key, value)` pairs and is useful for detecting duplicate keys.
- `parse_float`: controls how JSON floating-point literals are converted.
- `parse_int`: controls how JSON integer literals are converted.
- `parse_constant`: controls `NaN`, `Infinity`, and `-Infinity`.
- `decode`: receives the entire JSON document as text.
- `raw_decode`: decodes one JSON value from the beginning of a string and returns the ending index.


In [2]:
sample = '''
{
    "price": 10.25,
    "count": 3,
    "meta": {
        "active": true
    }
}
'''

print(json.loads(sample))
print(json.loads(sample, parse_float=Decimal))


{'price': 10.25, 'count': 3, 'meta': {'active': True}}
{'price': Decimal('10.25'), 'count': 3, 'meta': {'active': True}}


# Problem 1 — Build a Safe Tagged `Point` Decoder

You receive JSON where points are represented as:

```json
{"_type": "point", "x": 10, "y": 20}
```

Requirements:

1. Decode only dictionaries whose `_type` is exactly `"point"`.
2. Require exactly the fields `_type`, `x`, and `y`.
3. Reject booleans as coordinates, even though `bool` is a subclass of `int`.
4. Accept `int`, `float`, or `Decimal` coordinates.
5. Preserve all ordinary dictionaries unchanged.
6. Use `object_hook` instead of recursively walking the result after parsing.


In [3]:
@dataclass(frozen=True)
class Point:
    x: object
    y: object


def is_number_but_not_bool(value):
    return isinstance(value, (int, float, Decimal)) and not isinstance(value, bool)


def point_object_hook(obj):
    if obj.get("_type") != "point":
        return obj

    required = {"_type", "x", "y"}
    if set(obj) != required:
        raise ValueError(
            f"Invalid point keys: expected {required}, got {set(obj)}"
        )

    if not is_number_but_not_bool(obj["x"]):
        raise TypeError("Point.x must be numeric and must not be bool")

    if not is_number_but_not_bool(obj["y"]):
        raise TypeError("Point.y must be numeric and must not be bool")

    return Point(obj["x"], obj["y"])


In [4]:
document = '''
{
    "name": "triangle",
    "vertices": [
        {"_type": "point", "x": 0, "y": 0},
        {"_type": "point", "x": 2.5, "y": 0},
        {"_type": "point", "x": 1, "y": 3.75}
    ],
    "metadata": {
        "author": "Ada"
    }
}
'''

result = json.loads(
    document,
    object_hook=point_object_hook,
    parse_float=Decimal
)

pprint(result)
assert isinstance(result["vertices"][0], Point)
assert isinstance(result["vertices"][1].x, Decimal)
assert result["metadata"] == {"author": "Ada"}


{'metadata': {'author': 'Ada'},
 'name': 'triangle',
 'vertices': [Point(x=0, y=0),
              Point(x=Decimal('2.5'), y=0),
              Point(x=1, y=Decimal('3.75'))]}


### Solution discussion

`object_hook` is invoked on nested dictionaries before their containing dictionaries.
That means tagged inner objects can be converted automatically without writing a
manual recursive traversal.

This is usually cleaner and less error-prone than:

1. parsing to generic dictionaries,
2. scanning the entire result,
3. mutating nested containers afterward.


# Problem 2 — Design a Multi-Type Decoder Registry

Extend the tagged-object idea.

Supported JSON types:

- `point`
- `complex`
- `set`
- `tuple`

Examples:

```json
{"_type": "complex", "real": 2.5, "imag": -1}
{"_type": "set", "items": [1, 2, 3]}
{"_type": "tuple", "items": ["a", "b"]}
```

Requirements:

1. Avoid a long `if`/`elif` chain.
2. Use a registry that maps tag names to decoder functions.
3. Reject unknown `_type` values.
4. Decode nested tagged values correctly.


In [5]:
def decode_point(obj):
    expected = {"_type", "x", "y"}
    if set(obj) != expected:
        raise ValueError("Malformed point object")
    return Point(obj["x"], obj["y"])


def decode_complex(obj):
    expected = {"_type", "real", "imag"}
    if set(obj) != expected:
        raise ValueError("Malformed complex object")
    return complex(obj["real"], obj["imag"])


def decode_set(obj):
    expected = {"_type", "items"}
    if set(obj) != expected:
        raise ValueError("Malformed set object")
    return set(obj["items"])


def decode_tuple(obj):
    expected = {"_type", "items"}
    if set(obj) != expected:
        raise ValueError("Malformed tuple object")
    return tuple(obj["items"])


TYPE_DECODERS = {
    "point": decode_point,
    "complex": decode_complex,
    "set": decode_set,
    "tuple": decode_tuple,
}


def registry_object_hook(obj):
    tag = obj.get("_type")

    if tag is None:
        return obj

    try:
        decoder = TYPE_DECODERS[tag]
    except KeyError as exc:
        raise ValueError(f"Unknown tagged JSON type: {tag!r}") from exc

    return decoder(obj)


In [6]:
registry_json = '''
{
    "origin": {"_type": "point", "x": 0, "y": 0},
    "z": {"_type": "complex", "real": 2.5, "imag": -1},
    "labels": {"_type": "set", "items": ["red", "green", "blue"]},
    "shape": {
        "_type": "tuple",
        "items": [
            {"_type": "point", "x": 1, "y": 2},
            {"_type": "point", "x": 3, "y": 4}
        ]
    }
}
'''

decoded = json.loads(
    registry_json,
    object_hook=registry_object_hook,
    parse_float=Decimal
)

pprint(decoded)

assert decoded["origin"] == Point(0, 0)
assert decoded["z"] == complex(2.5, -1)
assert decoded["labels"] == {"red", "green", "blue"}
assert isinstance(decoded["shape"], tuple)
assert all(isinstance(item, Point) for item in decoded["shape"])


{'labels': {'blue', 'green', 'red'},
 'origin': Point(x=0, y=0),
 'shape': (Point(x=1, y=2), Point(x=3, y=4)),
 'z': (2.5-1j)}


# Problem 3 — Preserve Financial Precision with `Decimal`

A financial API returns:

```json
{
  "unit_price": 0.1,
  "quantity": 3,
  "tax_rate": 0.075
}
```

The naive floating-point computation may produce binary rounding artifacts.

Requirements:

1. Decode all JSON floating-point literals as `Decimal`.
2. Compute `subtotal`, `tax`, and `total`.
3. Quantize currency values to two decimal places.
4. Explain why constructing `Decimal` directly from a float is usually a mistake.


In [7]:
financial_json = '''
{
    "unit_price": 0.1,
    "quantity": 3,
    "tax_rate": 0.075
}
'''

financial = json.loads(financial_json, parse_float=Decimal)

subtotal = financial["unit_price"] * financial["quantity"]
tax = subtotal * financial["tax_rate"]
total = subtotal + tax

money = Decimal("0.01")

print("unit_price:", financial["unit_price"], type(financial["unit_price"]))
print("subtotal:", subtotal)
print("tax:", tax)
print("total:", total)
print("rounded total:", total.quantize(money))

assert financial["unit_price"] == Decimal("0.1")
assert total.quantize(money) == Decimal("0.32")


unit_price: 0.1 <class 'decimal.Decimal'>
subtotal: 0.3
tax: 0.0225
total: 0.3225
rounded total: 0.32


In [8]:
# Best practice:
good = Decimal("0.1")

# Usually undesirable:
bad = Decimal(0.1)

print("Decimal from string:", good)
print("Decimal from float :", bad)


Decimal from string: 0.1
Decimal from float : 0.1000000000000000055511151231257827021181583404541015625


### Why?

A JSON token such as `0.1` is textual. `parse_float=Decimal` lets `Decimal`
consume that text directly, preserving the decimal value the JSON author wrote.

By contrast, `Decimal(0.1)` first accepts the already-rounded binary floating-point
value represented by Python's `float`.


# Problem 4 — Reject Non-Standard Numeric Constants

Python's JSON decoder can recognize non-standard values such as:

- `NaN`
- `Infinity`
- `-Infinity`

Strict JSON systems often want to reject these.

Requirements:

1. Write a `parse_constant` function that raises an exception.
2. Verify that valid JSON numbers still decode.
3. Verify that `NaN`, `Infinity`, and `-Infinity` are rejected.


In [9]:
def reject_nonfinite_constant(token):
    raise ValueError(f"Non-standard JSON numeric constant is forbidden: {token}")


strict_numbers = json.JSONDecoder(
    parse_float=Decimal,
    parse_constant=reject_nonfinite_constant
)

print(strict_numbers.decode('{"x": 1.25, "y": 5}'))

for bad_json in ['{"x": NaN}', '{"x": Infinity}', '{"x": -Infinity}']:
    try:
        strict_numbers.decode(bad_json)
    except ValueError as exc:
        print("Rejected:", bad_json, "->", exc)


{'x': Decimal('1.25'), 'y': 5}
Rejected: {"x": NaN} -> Non-standard JSON numeric constant is forbidden: NaN
Rejected: {"x": Infinity} -> Non-standard JSON numeric constant is forbidden: Infinity
Rejected: {"x": -Infinity} -> Non-standard JSON numeric constant is forbidden: -Infinity


# Problem 5 — Detect Duplicate JSON Object Keys

JSON such as:

```json
{"role": "user", "role": "admin"}
```

is dangerous because ordinary decoding silently keeps the last value.

Requirements:

1. Use `object_pairs_hook`.
2. Raise `ValueError` on duplicate keys.
3. Preserve insertion order.
4. Test nested objects too.


In [10]:
def reject_duplicate_keys(pairs):
    result = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key detected: {key!r}")
        result[key] = value

    return result


In [11]:
good_json = '''
{
    "user": {
        "name": "Grace",
        "role": "admin"
    }
}
'''

print(json.loads(good_json, object_pairs_hook=reject_duplicate_keys))

duplicate_json = '''
{
    "user": {
        "name": "Grace",
        "role": "user",
        "role": "admin"
    }
}
'''

try:
    json.loads(duplicate_json, object_pairs_hook=reject_duplicate_keys)
except ValueError as exc:
    print("Rejected duplicate key:", exc)


{'user': {'name': 'Grace', 'role': 'admin'}}
Rejected duplicate key: Duplicate JSON key detected: 'role'


### Important precedence rule

If `object_pairs_hook` is supplied, it takes priority over `object_hook`.

If you need both duplicate-key detection **and** tagged-object conversion,
combine both behaviors inside the function supplied as `object_pairs_hook`.


# Problem 6 — Combine Duplicate-Key Detection with Tagged Objects

Requirements:

1. Reject duplicate keys.
2. Decode `_type: "point"` objects.
3. Decode floats as `Decimal`.
4. Keep ordinary JSON objects as dictionaries.


In [12]:
def pairs_to_typed_object(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key: {key!r}")
        obj[key] = value

    if obj.get("_type") == "point":
        if set(obj) != {"_type", "x", "y"}:
            raise ValueError("Malformed point")
        return Point(obj["x"], obj["y"])

    return obj


In [13]:
combined_json = '''
{
    "name": "segment",
    "start": {"_type": "point", "x": 0.1, "y": 0.2},
    "end": {"_type": "point", "x": 10.5, "y": 20.75}
}
'''

combined = json.loads(
    combined_json,
    object_pairs_hook=pairs_to_typed_object,
    parse_float=Decimal
)

pprint(combined)

assert combined["start"] == Point(Decimal("0.1"), Decimal("0.2"))
assert combined["end"] == Point(Decimal("10.5"), Decimal("20.75"))


{'end': Point(x=Decimal('10.5'), y=Decimal('20.75')),
 'name': 'segment',
 'start': Point(x=Decimal('0.1'), y=Decimal('0.2'))}


# Problem 7 — Decode `datetime` and `UUID` Objects

Tagged JSON:

```json
{"_type": "datetime", "value": "2026-08-07T12:30:00+00:00"}
{"_type": "uuid", "value": "550e8400-e29b-41d4-a716-446655440000"}
```

Requirements:

1. Convert datetimes with `datetime.fromisoformat`.
2. Reject naive datetimes that have no timezone.
3. Convert UUID strings with `UUID`.
4. Provide clear errors for malformed values.


In [14]:
def decode_datetime_object(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed datetime object")

    try:
        dt = datetime.fromisoformat(obj["value"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Invalid ISO-8601 datetime: {obj['value']!r}") from exc

    if dt.tzinfo is None:
        raise ValueError("Datetime must include timezone information")

    return dt


def decode_uuid_object(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed UUID object")

    try:
        return UUID(obj["value"])
    except (TypeError, ValueError, AttributeError) as exc:
        raise ValueError(f"Invalid UUID: {obj['value']!r}") from exc


EXTENDED_DECODERS = {
    **TYPE_DECODERS,
    "datetime": decode_datetime_object,
    "uuid": decode_uuid_object,
}


def extended_object_hook(obj):
    tag = obj.get("_type")
    if tag is None:
        return obj

    try:
        decoder = EXTENDED_DECODERS[tag]
    except KeyError as exc:
        raise ValueError(f"Unknown tagged type: {tag!r}") from exc

    return decoder(obj)


In [15]:
identifier = "550e8400-e29b-41d4-a716-446655440000"

event_json = f'''
{{
    "event_id": {{"_type": "uuid", "value": "{identifier}"}},
    "created_at": {{
        "_type": "datetime",
        "value": "2026-08-07T12:30:00+00:00"
    }},
    "location": {{"_type": "point", "x": 1.25, "y": -9.5}}
}}
'''

event = json.loads(
    event_json,
    object_hook=extended_object_hook,
    parse_float=Decimal
)

pprint(event)
assert isinstance(event["event_id"], UUID)
assert isinstance(event["created_at"], datetime)
assert event["created_at"].tzinfo is not None
assert event["location"] == Point(Decimal("1.25"), Decimal("-9.5"))


{'created_at': datetime.datetime(2026, 8, 7, 12, 30, tzinfo=datetime.timezone.utc),
 'event_id': UUID('550e8400-e29b-41d4-a716-446655440000'),
 'location': Point(x=Decimal('1.25'), y=Decimal('-9.5'))}


# Problem 8 — Build a Reusable `DomainJSONDecoder`

Create a `JSONDecoder` subclass that bundles:

- `parse_float=Decimal`
- strict rejection of `NaN` / infinities
- the extended tagged-object hook

Best-practice challenge:

Do **not** override `decode` merely to call `json.loads` again. Configure the
base `JSONDecoder` correctly through `super().__init__`.


In [16]:
class DomainJSONDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("parse_float", Decimal)
        kwargs.setdefault("parse_constant", reject_nonfinite_constant)
        kwargs.setdefault("object_hook", extended_object_hook)
        super().__init__(*args, **kwargs)


In [17]:
domain_json = '''
{
    "price": 99.95,
    "location": {"_type": "point", "x": 3.5, "y": 4.25},
    "when": {
        "_type": "datetime",
        "value": "2026-08-07T18:00:00+03:00"
    }
}
'''

domain_value = json.loads(domain_json, cls=DomainJSONDecoder)
pprint(domain_value)

assert domain_value["price"] == Decimal("99.95")
assert isinstance(domain_value["location"], Point)
assert isinstance(domain_value["when"], datetime)


{'location': Point(x=Decimal('3.5'), y=Decimal('4.25')),
 'price': Decimal('99.95'),
 'when': datetime.datetime(2026, 8, 7, 18, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800)))}


# Problem 9 — Schema-Style Validation During Decoding

Suppose a tagged `user` object must contain:

- `_type`
- `id`
- `name`
- `email`
- `roles`

Rules:

- `id` must be a positive integer but not a boolean.
- `name` must be a non-empty string.
- `email` must contain exactly one `@` for this exercise.
- `roles` must be a non-empty list of unique strings.

Create an immutable `User` dataclass and decode valid users.
Reject malformed users immediately.


In [18]:
@dataclass(frozen=True)
class User:
    id: int
    name: str
    email: str
    roles: tuple


def decode_user(obj):
    expected = {"_type", "id", "name", "email", "roles"}
    if set(obj) != expected:
        raise ValueError(f"Malformed user keys: {set(obj)}")

    user_id = obj["id"]
    name = obj["name"]
    email = obj["email"]
    roles = obj["roles"]

    if isinstance(user_id, bool) or not isinstance(user_id, int) or user_id <= 0:
        raise ValueError("User.id must be a positive integer")

    if not isinstance(name, str) or not name.strip():
        raise ValueError("User.name must be a non-empty string")

    if not isinstance(email, str) or email.count("@") != 1:
        raise ValueError("User.email is invalid for this exercise")

    if not isinstance(roles, list) or not roles:
        raise ValueError("User.roles must be a non-empty list")

    if not all(isinstance(role, str) and role for role in roles):
        raise ValueError("Every role must be a non-empty string")

    if len(set(roles)) != len(roles):
        raise ValueError("User.roles must not contain duplicates")

    return User(
        id=user_id,
        name=name.strip(),
        email=email,
        roles=tuple(roles)
    )


In [19]:
VALIDATED_DECODERS = {
    **EXTENDED_DECODERS,
    "user": decode_user,
}


def validated_object_hook(obj):
    tag = obj.get("_type")
    if tag is None:
        return obj

    try:
        decoder = VALIDATED_DECODERS[tag]
    except KeyError as exc:
        raise ValueError(f"Unknown tagged type: {tag!r}") from exc

    return decoder(obj)


In [20]:
user_json = '''
{
    "_type": "user",
    "id": 42,
    "name": "  Linus  ",
    "email": "linus@example.com",
    "roles": ["maintainer", "reviewer"]
}
'''

user = json.loads(user_json, object_hook=validated_object_hook)
print(user)

assert user == User(
    id=42,
    name="Linus",
    email="linus@example.com",
    roles=("maintainer", "reviewer")
)


User(id=42, name='Linus', email='linus@example.com', roles=('maintainer', 'reviewer'))


In [21]:
bad_users = [
    '{"_type":"user","id":true,"name":"A","email":"a@b","roles":["user"]}',
    '{"_type":"user","id":1,"name":"","email":"a@b","roles":["user"]}',
    '{"_type":"user","id":1,"name":"A","email":"invalid","roles":["user"]}',
    '{"_type":"user","id":1,"name":"A","email":"a@b","roles":["user","user"]}',
]

for text in bad_users:
    try:
        json.loads(text, object_hook=validated_object_hook)
    except ValueError as exc:
        print("Rejected:", exc)


Rejected: User.id must be a positive integer
Rejected: User.name must be a non-empty string
Rejected: User.email is invalid for this exercise
Rejected: User.roles must not contain duplicates


# Problem 10 — Decode Multiple Concatenated JSON Values with `raw_decode`

You receive a buffer like:

```text
{"id":1} {"id":2}
[10, 20, 30]
"done"
```

This is **not one valid JSON document**, but it contains several valid JSON values.

Requirements:

1. Use `JSONDecoder.raw_decode`.
2. Skip whitespace between values.
3. Return all decoded values.
4. Raise a useful error if malformed data appears.


In [22]:
def decode_many(text, decoder=None):
    decoder = decoder or json.JSONDecoder()
    values = []
    index = 0
    length = len(text)

    while index < length:
        while index < length and text[index].isspace():
            index += 1

        if index >= length:
            break

        try:
            value, end = decoder.raw_decode(text, index)
        except json.JSONDecodeError as exc:
            snippet = text[index:index + 40]
            raise ValueError(
                f"Invalid JSON near index {index}: {snippet!r}"
            ) from exc

        values.append(value)
        index = end

    return values


In [23]:
buffer = '''
{"id": 1}
{"id": 2, "position": {"_type": "point", "x": 1.5, "y": 2.5}}
[10, 20, 30]
"done"
'''

values = decode_many(buffer, DomainJSONDecoder())
pprint(values)

assert values[0] == {"id": 1}
assert values[1]["position"] == Point(Decimal("1.5"), Decimal("2.5"))
assert values[2] == [10, 20, 30]
assert values[3] == "done"


[{'id': 1},
 {'id': 2, 'position': Point(x=Decimal('1.5'), y=Decimal('2.5'))},
 [10, 20, 30],
 'done']


# Problem 11 — Implement Bounded Integer Parsing

An untrusted payload could contain extremely large integer literals.

For this exercise, allow only integers in the inclusive range:

```text
[-10^12, 10^12]
```

Requirements:

1. Use `parse_int`.
2. Parse from the original string token.
3. Reject out-of-range values.
4. Keep normal integers as Python `int`.


In [24]:
MAX_ABS_INT = 10**12


def bounded_int(token):
    value = int(token)

    if not -MAX_ABS_INT <= value <= MAX_ABS_INT:
        raise ValueError(
            f"Integer {value} is outside allowed range "
            f"[-{MAX_ABS_INT}, {MAX_ABS_INT}]"
        )

    return value


In [25]:
bounded_decoder = json.JSONDecoder(parse_int=bounded_int)

print(bounded_decoder.decode('{"n": 999999999999}'))

try:
    bounded_decoder.decode('{"n": 999999999999999999999999999}')
except ValueError as exc:
    print("Rejected oversized integer:", exc)


{'n': 999999999999}
Rejected oversized integer: Integer 999999999999999999999999999 is outside allowed range [-1000000000000, 1000000000000]


# Problem 12 — Round-Trip Domain Objects

Build a matching encoder and decoder for:

- `Point`
- `datetime`
- `UUID`
- `Decimal`
- `set`
- `tuple`

Requirements:

1. Encode objects using explicit `_type` tags.
2. Decode them back to useful Python objects.
3. Keep the format readable.
4. Verify a nested round trip.
5. Handle an important `JSONEncoder` nuance: `default()` is **not called for tuples**
   because the standard encoder already knows how to serialize tuples as JSON arrays.
   Therefore, preserving tuple identity requires a pre-processing step that tags tuples
   before `json.dumps` sees them.


In [26]:
class DomainJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Point):
            return {
                "_type": "point",
                "x": obj.x,
                "y": obj.y,
            }

        if isinstance(obj, datetime):
            if obj.tzinfo is None:
                raise TypeError("Refusing to encode naive datetime")
            return {
                "_type": "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, UUID):
            return {
                "_type": "uuid",
                "value": str(obj),
            }

        if isinstance(obj, Decimal):
            # Standard JSON has no native exact decimal type.
            return {
                "_type": "decimal",
                "value": str(obj),
            }

        if isinstance(obj, set):
            return {
                "_type": "set",
                "items": sorted(obj, key=repr),
            }

        # Do not try to preserve tuple identity here:
        # JSONEncoder.default() is not called for tuples.
        return super().default(obj)


def prepare_json_value(value):
    """Recursively tag tuples before the standard encoder turns them into arrays."""
    if isinstance(value, tuple):
        return {
            "_type": "tuple",
            "items": [prepare_json_value(item) for item in value],
        }

    if isinstance(value, list):
        return [prepare_json_value(item) for item in value]

    if isinstance(value, dict):
        return {
            key: prepare_json_value(item)
            for key, item in value.items()
        }

    return value


In [27]:
def decode_decimal_object(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed decimal object")

    try:
        return Decimal(obj["value"])
    except Exception as exc:
        raise ValueError(f"Invalid Decimal value: {obj['value']!r}") from exc


ROUNDTRIP_DECODERS = {
    **EXTENDED_DECODERS,
    "decimal": decode_decimal_object,
}


def roundtrip_hook(obj):
    tag = obj.get("_type")
    if tag is None:
        return obj

    try:
        decoder = ROUNDTRIP_DECODERS[tag]
    except KeyError as exc:
        raise ValueError(f"Unknown tagged type: {tag!r}") from exc

    return decoder(obj)


In [28]:
payload = {
    "point": Point(Decimal("1.1"), Decimal("2.2")),
    "created": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    "id": UUID("550e8400-e29b-41d4-a716-446655440000"),
    "amount": Decimal("1234.5678"),
    "tags": {"json", "python"},
    "coords": (10, 20),
}

prepared_payload = prepare_json_value(payload)

encoded = json.dumps(
    prepared_payload,
    cls=DomainJSONEncoder,
    indent=2,
    sort_keys=True
)

print(encoded)

decoded = json.loads(
    encoded,
    object_hook=roundtrip_hook
)

pprint(decoded)

assert decoded["point"] == payload["point"]
assert decoded["created"] == payload["created"]
assert decoded["id"] == payload["id"]
assert decoded["amount"] == payload["amount"]
assert decoded["tags"] == payload["tags"]
assert decoded["coords"] == payload["coords"]


{
  "amount": {
    "_type": "decimal",
    "value": "1234.5678"
  },
  "coords": {
    "_type": "tuple",
    "items": [
      10,
      20
    ]
  },
  "created": {
    "_type": "datetime",
    "value": "2026-08-07T12:00:00+00:00"
  },
  "id": {
    "_type": "uuid",
    "value": "550e8400-e29b-41d4-a716-446655440000"
  },
  "point": {
    "_type": "point",
    "x": {
      "_type": "decimal",
      "value": "1.1"
    },
    "y": {
      "_type": "decimal",
      "value": "2.2"
    }
  },
  "tags": {
    "_type": "set",
    "items": [
      "json",
      "python"
    ]
  }
}
{'amount': Decimal('1234.5678'),
 'coords': (10, 20),
 'created': datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc),
 'id': UUID('550e8400-e29b-41d4-a716-446655440000'),
 'point': Point(x=Decimal('1.1'), y=Decimal('2.2')),
 'tags': {'json', 'python'}}


# Problem 13 — Versioned JSON Types

Long-lived data formats evolve.

Suppose point version 1 used:

```json
{"_type": "point", "_version": 1, "x": 1, "y": 2}
```

but version 2 uses:

```json
{"_type": "point", "_version": 2, "coordinates": [1, 2]}
```

Requirements:

1. Support both versions.
2. Reject unsupported versions.
3. Decode both to the same `Point` class.
4. Avoid guessing when `_version` is missing.


In [29]:
def decode_versioned_point(obj):
    version = obj.get("_version")

    if version == 1:
        expected = {"_type", "_version", "x", "y"}
        if set(obj) != expected:
            raise ValueError("Malformed point v1")
        return Point(obj["x"], obj["y"])

    if version == 2:
        expected = {"_type", "_version", "coordinates"}
        if set(obj) != expected:
            raise ValueError("Malformed point v2")

        coordinates = obj["coordinates"]
        if not isinstance(coordinates, list) or len(coordinates) != 2:
            raise ValueError("point v2 coordinates must contain exactly 2 items")

        return Point(*coordinates)

    raise ValueError(f"Unsupported or missing point version: {version!r}")


In [30]:
def versioned_hook(obj):
    if obj.get("_type") == "point":
        return decode_versioned_point(obj)
    return obj


v1 = '{"_type":"point","_version":1,"x":10,"y":20}'
v2 = '{"_type":"point","_version":2,"coordinates":[10,20]}'

assert json.loads(v1, object_hook=versioned_hook) == Point(10, 20)
assert json.loads(v2, object_hook=versioned_hook) == Point(10, 20)

print(json.loads(v1, object_hook=versioned_hook))
print(json.loads(v2, object_hook=versioned_hook))


Point(x=10, y=20)
Point(x=10, y=20)


# Problem 14 — Whole-Document Validation by Overriding `decode`

This is a case where overriding `decode` is justified.

Every accepted document must:

1. Decode to a dictionary.
2. Contain top-level keys `schema_version` and `payload`.
3. Have `schema_version == 1`.
4. Use all standard domain decoding features for nested content.

Implement a decoder that validates the complete decoded document after the base
decoder finishes.


In [31]:
class EnvelopeDecoder(DomainJSONDecoder):
    def decode(self, s, _w=json.decoder.WHITESPACE.match):
        obj = super().decode(s, _w)

        if not isinstance(obj, dict):
            raise ValueError("Top-level JSON value must be an object")

        required = {"schema_version", "payload"}
        if not required.issubset(obj):
            missing = required - set(obj)
            raise ValueError(f"Missing top-level keys: {sorted(missing)}")

        if obj["schema_version"] != 1:
            raise ValueError(
                f"Unsupported schema_version: {obj['schema_version']!r}"
            )

        return obj


In [32]:
envelope_json = '''
{
    "schema_version": 1,
    "payload": {
        "price": 12.50,
        "location": {
            "_type": "point",
            "x": 4.25,
            "y": 8.75
        }
    }
}
'''

envelope = json.loads(envelope_json, cls=EnvelopeDecoder)
pprint(envelope)

assert envelope["payload"]["price"] == Decimal("12.50")
assert envelope["payload"]["location"] == Point(
    Decimal("4.25"),
    Decimal("8.75")
)


{'payload': {'location': Point(x=Decimal('4.25'), y=Decimal('8.75')),
             'price': Decimal('12.50')},
 'schema_version': 1}


# Problem 15 — A Decoder Factory for Different Trust Levels

Create a factory with two modes:

### `"trusted"`

- `Decimal` floats
- tagged object conversion
- allows large integers

### `"strict"`

- everything from trusted mode
- rejects duplicate keys
- rejects `NaN` and infinities
- bounds integers to `10^12`

Because `object_pairs_hook` takes priority over `object_hook`, the strict mode
must combine duplicate checking and tagged conversion.


In [33]:
def strict_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key: {key!r}")
        obj[key] = value

    return extended_object_hook(obj)


def make_decoder(mode):
    if mode == "trusted":
        return json.JSONDecoder(
            parse_float=Decimal,
            object_hook=extended_object_hook
        )

    if mode == "strict":
        return json.JSONDecoder(
            parse_float=Decimal,
            parse_int=bounded_int,
            parse_constant=reject_nonfinite_constant,
            object_pairs_hook=strict_pairs_hook
        )

    raise ValueError(f"Unknown decoder mode: {mode!r}")


In [34]:
trusted = make_decoder("trusted")
strict = make_decoder("strict")

example = '''
{
    "value": 1.25,
    "point": {"_type": "point", "x": 2.5, "y": 3.5}
}
'''

print("trusted:")
pprint(trusted.decode(example))

print("\nstrict:")
pprint(strict.decode(example))


trusted:
{'point': Point(x=Decimal('2.5'), y=Decimal('3.5')), 'value': Decimal('1.25')}

strict:
{'point': Point(x=Decimal('2.5'), y=Decimal('3.5')), 'value': Decimal('1.25')}


# Problem 16 — Decode a Recursive Expression Tree

Represent an arithmetic expression in JSON:

```json
{
  "_type": "binary_op",
  "op": "+",
  "left": {"_type": "number", "value": 10},
  "right": {
    "_type": "binary_op",
    "op": "*",
    "left": {"_type": "number", "value": 2},
    "right": {"_type": "number", "value": 3}
  }
}
```

Requirements:

1. Decode into immutable dataclasses.
2. Allow operators `+`, `-`, `*`, `/`.
3. Reject unknown operators.
4. Evaluate the reconstructed expression tree.
5. Notice that `object_hook` naturally reconstructs children before parents.


In [35]:
@dataclass(frozen=True)
class Number:
    value: object


@dataclass(frozen=True)
class BinaryOp:
    op: str
    left: object
    right: object


def expression_hook(obj):
    tag = obj.get("_type")

    if tag == "number":
        if set(obj) != {"_type", "value"}:
            raise ValueError("Malformed number node")
        return Number(obj["value"])

    if tag == "binary_op":
        if set(obj) != {"_type", "op", "left", "right"}:
            raise ValueError("Malformed binary_op node")

        if obj["op"] not in {"+", "-", "*", "/"}:
            raise ValueError(f"Unsupported operator: {obj['op']!r}")

        return BinaryOp(
            op=obj["op"],
            left=obj["left"],
            right=obj["right"]
        )

    return obj


def evaluate(node):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left)
        right = evaluate(node.right)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

    raise TypeError(f"Unsupported expression node: {node!r}")


In [36]:
expression_json = '''
{
    "_type": "binary_op",
    "op": "+",
    "left": {"_type": "number", "value": 10},
    "right": {
        "_type": "binary_op",
        "op": "*",
        "left": {"_type": "number", "value": 2.5},
        "right": {"_type": "number", "value": 3}
    }
}
'''

tree = json.loads(
    expression_json,
    object_hook=expression_hook,
    parse_float=Decimal
)

print(tree)
print("result:", evaluate(tree))

assert evaluate(tree) == Decimal("17.5")


BinaryOp(op='+', left=Number(value=10), right=BinaryOp(op='*', left=Number(value=Decimal('2.5')), right=Number(value=3)))
result: 17.5


# Problem 17 — Fail Closed on Suspicious Type Tags

A subtle security/design problem:

Should this ordinary business object:

```json
{"_type": "point", "x": "not-a-number", "y": 2}
```

quietly remain a dictionary if decoding fails?

For a tagged protocol, the safer design is often **fail closed**:

- if `_type` is present, treat the object as intentionally typed;
- if the tag is unknown or malformed, raise;
- do not silently ignore malformed tagged data.

Write tests proving this behavior.


In [37]:
suspicious_payloads = [
    '{"_type":"point","x":"oops","y":2}',
    '{"_type":"does_not_exist","value":123}',
    '{"_type":"uuid","value":"not-a-uuid"}',
]

for payload in suspicious_payloads:
    try:
        json.loads(payload, object_hook=extended_object_hook)
    except (ValueError, TypeError) as exc:
        print("Correctly rejected:", payload)
        print("Reason:", exc)
        print("-" * 60)


Correctly rejected: {"_type":"does_not_exist","value":123}
Reason: Unknown tagged type: 'does_not_exist'
------------------------------------------------------------
Correctly rejected: {"_type":"uuid","value":"not-a-uuid"}
Reason: Invalid UUID: 'not-a-uuid'
------------------------------------------------------------


# Problem 18 — JSON Lines (`.jsonl`) with Per-Line Error Reporting

JSON Lines stores one complete JSON value per line.

Requirements:

1. Decode each non-blank line separately.
2. Use `DomainJSONDecoder`.
3. Return valid objects.
4. Collect errors with line numbers instead of stopping at the first bad line.


In [38]:
def decode_json_lines(text, decoder=None):
    decoder = decoder or DomainJSONDecoder()

    values = []
    errors = []

    for line_number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue

        try:
            values.append(decoder.decode(line))
        except Exception as exc:
            errors.append({
                "line": line_number,
                "text": line,
                "error": str(exc),
            })

    return values, errors


In [39]:
jsonl = '''
{"id": 1, "amount": 1.25}
{"id": 2, "point": {"_type": "point", "x": 3.5, "y": 4.5}}
{"id": 3, "broken":
{"id": 4, "amount": 9.99}
'''

values, errors = decode_json_lines(jsonl)

print("VALID VALUES")
pprint(values)

print("\nERRORS")
pprint(errors)

assert len(values) == 3
assert len(errors) == 1


VALID VALUES
[{'amount': Decimal('1.25'), 'id': 1},
 {'id': 2, 'point': Point(x=Decimal('3.5'), y=Decimal('4.5'))},
 {'amount': Decimal('9.99'), 'id': 4}]

ERRORS
[{'error': 'Expecting value: line 1 column 20 (char 19)',
  'line': 4,
  'text': '{"id": 3, "broken":'}]


# Problem 19 — Custom Integer Type: Preserve Very Large IDs as Strings

Some APIs return numeric-looking identifiers such as:

```json
{"account_id": 123456789012345678901234567890}
```

You may want to avoid treating every huge token as a numeric quantity.

For this exercise:

- integers with at most 15 digits become `int`;
- larger integer literals become strings.

Implement this with `parse_int`.


In [40]:
def int_or_string(token):
    digits = token.lstrip("-")

    if len(digits) <= 15:
        return int(token)

    return token


In [41]:
id_json = '''
{
    "small": 12345,
    "huge": 123456789012345678901234567890
}
'''

ids = json.loads(id_json, parse_int=int_or_string)

print(ids)
print(type(ids["small"]))
print(type(ids["huge"]))

assert ids["small"] == 12345
assert ids["huge"] == "123456789012345678901234567890"


{'small': 12345, 'huge': '123456789012345678901234567890'}
<class 'int'>
<class 'str'>


# Problem 20 — Cap Nesting Depth After Decoding

Deeply nested JSON can be expensive to process.

The standard `json` API does not expose a simple `max_depth` argument.

For this exercise:

1. Decode normally.
2. Traverse the result iteratively.
3. Reject structures deeper than a configured limit.
4. Avoid recursive Python traversal in the validator itself.


In [42]:
def validate_max_depth(root, max_depth):
    stack = [(root, 0)]

    while stack:
        value, depth = stack.pop()

        if depth > max_depth:
            raise ValueError(
                f"Maximum nesting depth exceeded: "
                f"{depth} > {max_depth}"
            )

        if isinstance(value, dict):
            for child in value.values():
                stack.append((child, depth + 1))

        elif isinstance(value, (list, tuple)):
            for child in value:
                stack.append((child, depth + 1))

    return root


def loads_with_depth_limit(text, max_depth=20, **kwargs):
    obj = json.loads(text, **kwargs)
    return validate_max_depth(obj, max_depth)


In [43]:
shallow = '{"a": {"b": [1, 2, 3]}}'
print(loads_with_depth_limit(shallow, max_depth=5))

deep = "0"
for _ in range(8):
    deep = "[" + deep + "]"

try:
    loads_with_depth_limit(deep, max_depth=5)
except ValueError as exc:
    print("Rejected deep structure:", exc)


{'a': {'b': [1, 2, 3]}}
Rejected deep structure: Maximum nesting depth exceeded: 6 > 5


# Capstone Problem — Production-Style Protocol Decoder

Build one decoder for a fictional event protocol.

Protocol rules:

- Top-level object only.
- `schema_version` must be `2`.
- Duplicate keys are forbidden.
- Floats become `Decimal`.
- Integers are bounded to `10^12`.
- `NaN` and infinities are rejected.
- Known tagged types:
  - `point`
  - `datetime`
  - `uuid`
  - `decimal`
  - `set`
  - `tuple`
  - `user`
- Unknown tags are errors.
- The top-level object must contain:
  - `schema_version`
  - `event_id`
  - `created_at`
  - `actor`
  - `payload`

This combines several techniques from the notebook.


In [44]:
PROTOCOL_DECODERS = {
    **ROUNDTRIP_DECODERS,
    "user": decode_user,
}


def protocol_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        obj[key] = value

    tag = obj.get("_type")
    if tag is None:
        return obj

    try:
        decoder = PROTOCOL_DECODERS[tag]
    except KeyError as exc:
        raise ValueError(f"Unknown protocol type: {tag!r}") from exc

    return decoder(obj)


class ProtocolDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs["parse_float"] = Decimal
        kwargs["parse_int"] = bounded_int
        kwargs["parse_constant"] = reject_nonfinite_constant
        kwargs["object_pairs_hook"] = protocol_pairs_hook

        # object_pairs_hook takes precedence over object_hook,
        # so tagged conversion is included in protocol_pairs_hook.
        kwargs.pop("object_hook", None)

        super().__init__(*args, **kwargs)

    def decode(self, s, _w=json.decoder.WHITESPACE.match):
        obj = super().decode(s, _w)

        if not isinstance(obj, dict):
            raise ValueError("Protocol document must be a JSON object")

        required = {
            "schema_version",
            "event_id",
            "created_at",
            "actor",
            "payload",
        }

        missing = required - set(obj)
        if missing:
            raise ValueError(f"Missing protocol fields: {sorted(missing)}")

        if obj["schema_version"] != 2:
            raise ValueError(
                f"Unsupported schema_version: {obj['schema_version']!r}"
            )

        if not isinstance(obj["event_id"], UUID):
            raise TypeError("event_id must decode to UUID")

        if not isinstance(obj["created_at"], datetime):
            raise TypeError("created_at must decode to datetime")

        if not isinstance(obj["actor"], User):
            raise TypeError("actor must decode to User")

        return obj


In [45]:
protocol_json = '''
{
    "schema_version": 2,

    "event_id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T15:00:00+00:00"
    },

    "actor": {
        "_type": "user",
        "id": 7,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin", "author"]
    },

    "payload": {
        "amount": {
            "_type": "decimal",
            "value": "1250.75"
        },

        "location": {
            "_type": "point",
            "x": 23.3219,
            "y": 42.6977
        },

        "flags": {
            "_type": "set",
            "items": ["priority", "audited"]
        },

        "dimensions": {
            "_type": "tuple",
            "items": [1920, 1080]
        }
    }
}
'''

protocol = json.loads(protocol_json, cls=ProtocolDecoder)
pprint(protocol)

assert protocol["schema_version"] == 2
assert isinstance(protocol["event_id"], UUID)
assert isinstance(protocol["created_at"], datetime)
assert isinstance(protocol["actor"], User)
assert protocol["payload"]["amount"] == Decimal("1250.75")
assert isinstance(protocol["payload"]["location"], Point)
assert protocol["payload"]["flags"] == {"priority", "audited"}
assert protocol["payload"]["dimensions"] == (1920, 1080)


{'actor': User(id=7,
               name='Ada',
               email='ada@example.com',
               roles=('admin', 'author')),
 'created_at': datetime.datetime(2026, 8, 7, 15, 0, tzinfo=datetime.timezone.utc),
 'event_id': UUID('550e8400-e29b-41d4-a716-446655440000'),
 'payload': {'amount': Decimal('1250.75'),
             'dimensions': (1920, 1080),
             'flags': {'priority', 'audited'},
             'location': Point(x=Decimal('23.3219'), y=Decimal('42.6977'))},
 'schema_version': 2}


## Capstone Negative Tests


In [46]:
bad_protocol_documents = {
    "duplicate key": '''
        {
            "schema_version": 2,
            "schema_version": 3,
            "event_id": {"_type":"uuid","value":"550e8400-e29b-41d4-a716-446655440000"},
            "created_at": {"_type":"datetime","value":"2026-08-07T15:00:00+00:00"},
            "actor": {"_type":"user","id":1,"name":"A","email":"a@b","roles":["user"]},
            "payload": {}
        }
    ''',

    "wrong schema": '''
        {
            "schema_version": 99,
            "event_id": {"_type":"uuid","value":"550e8400-e29b-41d4-a716-446655440000"},
            "created_at": {"_type":"datetime","value":"2026-08-07T15:00:00+00:00"},
            "actor": {"_type":"user","id":1,"name":"A","email":"a@b","roles":["user"]},
            "payload": {}
        }
    ''',

    "unknown type": '''
        {
            "schema_version": 2,
            "event_id": {"_type":"uuid","value":"550e8400-e29b-41d4-a716-446655440000"},
            "created_at": {"_type":"datetime","value":"2026-08-07T15:00:00+00:00"},
            "actor": {"_type":"user","id":1,"name":"A","email":"a@b","roles":["user"]},
            "payload": {"x":{"_type":"mystery","value":123}}
        }
    ''',

    "non-finite number": '''
        {
            "schema_version": 2,
            "event_id": {"_type":"uuid","value":"550e8400-e29b-41d4-a716-446655440000"},
            "created_at": {"_type":"datetime","value":"2026-08-07T15:00:00+00:00"},
            "actor": {"_type":"user","id":1,"name":"A","email":"a@b","roles":["user"]},
            "payload": {"score": NaN}
        }
    '''
}

for name, text in bad_protocol_documents.items():
    try:
        json.loads(text, cls=ProtocolDecoder)
    except Exception as exc:
        print(f"{name}: {type(exc).__name__}: {exc}")


duplicate key: ValueError: Duplicate JSON key: 'schema_version'
wrong schema: ValueError: Unsupported schema_version: 99
unknown type: ValueError: Unknown protocol type: 'mystery'
non-finite number: ValueError: Non-standard JSON numeric constant is forbidden: NaN


# Extra Challenge 1 — Path-Aware Validation

`object_hook` does not tell you the path of the object being decoded.

Design a two-stage approach:

1. Decode tagged objects normally.
2. Traverse the final structure while tracking paths such as:
   - `$.payload.location`
   - `$.users[3].email`
3. Report validation errors with paths.

A starter implementation follows.


In [47]:
def walk_with_paths(value, path="$"):
    yield path, value

    if isinstance(value, dict):
        for key, child in value.items():
            yield from walk_with_paths(
                child,
                f"{path}.{key}"
            )

    elif isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            yield from walk_with_paths(
                child,
                f"{path}[{index}]"
            )


example = {
    "payload": {
        "points": [
            Point(1, 2),
            Point(3, 4),
        ]
    }
}

for path, value in walk_with_paths(example):
    print(path, "->", value)


$ -> {'payload': {'points': [Point(x=1, y=2), Point(x=3, y=4)]}}
$.payload -> {'points': [Point(x=1, y=2), Point(x=3, y=4)]}
$.payload.points -> [Point(x=1, y=2), Point(x=3, y=4)]
$.payload.points[0] -> Point(x=1, y=2)
$.payload.points[1] -> Point(x=3, y=4)


# Extra Challenge 2 — `JSONDecodeError` Diagnostics

When syntax is invalid, `json.loads` raises `json.JSONDecodeError`.

Useful attributes include:

- `msg`
- `doc`
- `pos`
- `lineno`
- `colno`

Write a friendly diagnostic helper.


In [48]:
def explain_json_error(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        lines = text.splitlines()

        offending_line = ""
        if 1 <= exc.lineno <= len(lines):
            offending_line = lines[exc.lineno - 1]

        pointer = " " * max(exc.colno - 1, 0) + "^"

        return (
            f"{exc.msg}\n"
            f"line={exc.lineno}, column={exc.colno}, position={exc.pos}\n"
            f"{offending_line}\n"
            f"{pointer}"
        )


broken = '''
{
    "a": 1,
    "b": [10, 20,]
}
'''

print(explain_json_error(broken))


Illegal trailing comma before end of array
line=4, column=17, position=31
    "b": [10, 20,]
                ^


# Extra Challenge 3 — Benchmark Hook Strategies

For performance-sensitive code, compare:

1. ordinary `json.loads`
2. `object_hook`
3. a custom `JSONDecoder`
4. post-processing after ordinary decoding

Use `timeit` with realistic payloads.

Do not optimize based only on tiny toy JSON documents.


In [49]:
import timeit

benchmark_payload = json.dumps({
    "items": [
        {
            "id": i,
            "point": {
                "_type": "point",
                "x": i + 0.1,
                "y": i + 0.2,
            }
        }
        for i in range(500)
    ]
})


def plain_decode():
    return json.loads(benchmark_payload)


def hooked_decode():
    return json.loads(
        benchmark_payload,
        object_hook=extended_object_hook,
        parse_float=Decimal
    )


plain_time = timeit.timeit(plain_decode, number=20)
hooked_time = timeit.timeit(hooked_decode, number=20)

print(f"plain : {plain_time:.6f}s")
print(f"hooked: {hooked_time:.6f}s")
print(f"ratio : {hooked_time / plain_time:.2f}x")


plain : 0.018560s
hooked: 0.037963s
ratio : 2.05x


# Best-Practice Summary

1. **Prefer `object_hook` for tagged object reconstruction.**
   It naturally handles nested dictionaries bottom-up.

2. **Use `object_pairs_hook` when duplicate keys matter.**
   Ordinary dictionaries cannot tell you that a duplicate appeared because the
   duplicate has already been overwritten.

3. **Use `parse_float=Decimal` for decimal-sensitive domains.**
   Finance is the classic example.

4. **Use `parse_constant` for strict numeric policies.**
   Reject `NaN` and infinities when your protocol requires strict JSON.

5. **Use `parse_int` for custom integer policy.**
   You can bound, transform, or classify integer tokens.

6. **Fail closed for explicit type tags.**
   If `_type` exists but is unknown or malformed, silently keeping the object as
   a normal dictionary can hide data-contract problems.

7. **Validate exact fields for protocol objects.**
   This catches typos, unexpected fields, and accidental format drift.

8. **Version long-lived tagged formats.**
   Never depend on guessing which historical shape an object uses.

9. **Override `decode` only for whole-document behavior.**
   Examples: envelope validation, global schema checks, document-level invariants.

10. **Use `raw_decode` for multiple adjacent JSON values.**
    It is especially useful for buffers and parsers that need to consume one
    JSON value at a time.

11. **Keep JSON decoding and business validation conceptually separate when useful.**
    Parsing answers "what Python values are represented?" while validation answers
    "are these values acceptable for this application?"

12. **Do not treat JSON as a trusted object-instantiation protocol.**
    Only map tags to explicitly approved constructors. Never dynamically import or
    execute classes/functions named by untrusted JSON.


# Final Practice Set — Without Immediate Solutions

Try these independently before reviewing the earlier patterns.

### A.
Add a tagged `date` type that accepts only `YYYY-MM-DD`.

### B.
Add a tagged `money` type with:
- ISO currency code,
- exact `Decimal` amount,
- validation that currency is exactly 3 uppercase ASCII letters.

### C.
Add a tagged `matrix` type and validate:
- rectangular rows,
- numeric cells,
- non-empty dimensions.

### D.
Extend `decode_many` so it returns:
- decoded values,
- each value's start index,
- each value's end index.

### E.
Create an `object_pairs_hook` that:
- rejects duplicate keys case-insensitively,
- but preserves the spelling of the first occurrence.

### F.
Create a strict decoder that rejects integers with more than 100 digits
*before* converting them to `int`.

### G.
Create a versioned `user` decoder supporting v1 and v2, with a migration to
one current `User` dataclass.

### H.
Build a round-trip test suite using `assert` for every tagged domain type.

### I.
Write a decoder that permits unknown tags only under `$.metadata`.

### J.
Write property-style tests that generate many `Point` values, encode them,
decode them, and verify equality.


In [50]:
# Workspace for the final practice set.

# Example starter:
#
# def decode_date_object(obj):
#     ...
#
# Add your solutions below.
